# Theory

Irradiance of direct light on a plane depends on angle of incidence,
$$
\begin{align}
&N & \textrm{direct normal irradiance} \\
&\theta_{i} & \textrm{incidence angle} \\
&\theta_{zenith} & \textrm{zenith angle, incidence angle when instrument horizontal} \\
\mu_i &= \cos(\theta_{i})               & \textrm{projection factor}  \\
\mu_0 &= \cos(\theta_{zenith})          & \textrm{projection factor for hozizontal instrument}  \\
\mu_t & = \cos(\theta_{zenith}, \textrm{pitch, roll, heading}) & \textrm{projection factor of tilted platform}\\
I(\mu_i) &= N \mu_i & \textrm{observed direct irradiance} \\
I(\mu_i) = N \mu_i &\neq N \mu_0 = I(\mu_0), & \text{for } \mu_i \neq \mu_0 
\end{align}
$$
where $I$ is the irradiance at the incidence angle $\theta_i$ and $N$ is the direct normal irradiance. Often the projection factor $\mu_i$ is used rather then the incidence angle. When the instrument is horizontal $\theta_i$ is equal to the solar thenith angle and the projection factor is $\mu_0$

$$
\begin{align}
\end{align}
$$

Irradiance of diffuse radiation is less effected by the tilt, but gets very complicated when treated precisly.
Here we assume the simmples approximation --> The diffuse radiation is isotropic --> diffuse is not effected by tilt. Even this apprimation breaks down the more ground comes into few with increasing tilt.

![Tilt illustration](tilt_illustration.png)

## Solving the linear equations getting tilt corrected direct and global

In [13]:
import sympy as sp

$$
\begin{aligned}
&D & \textrm{diffuse irradiance} \\
G_t &= N\mu_t + D & \textrm{global irradiance of tilted plattform}\\ 
G_0 &= N\mu_0 + D & \textrm{global horizontal irradiance} \\
f_D &= \frac{D}{G_t} &\textrm{diffuse fraction of titlted global irradiance}
\end{aligned}
$$

Solving the equation system for the unknowns $G_0$, $N$, and $D$, in terms of the remaining variables $G_t$, $\mu_t$, $\mu_0$, and $f_D$, yields

$$
\begin{aligned}
G_0 &= G_t~\frac{-\mathrm{f_D}~\mu_0 + \mathrm{f_D}~\mu_t + \mu_0}{\mu_t}\\
N &= G_t~\frac{1 -\mathrm{f_D}}{\mu_t} \\
D &= G_t~\mathrm{f_D}
\end{aligned}
$$

In [14]:
# based on the ratio of diffuse and global. This is preferable as it uses two measurments that are actually observed
Gt, N, mut, mu0, Diff, G0, ddt = sp.symbols("Gt N mut mu0 Diff G0 ddt") 

#equations
eqs = [sp.Eq(Gt, N * mut + Diff), 
       sp.Eq(G0, N * mu0 + Diff), 
       sp.Eq(ddt, Diff/Gt), ] 
sol = sp.solve(eqs, [G0, N,Diff], dict=True)
sol

[{Diff: Gt*ddt,
  G0: (-Gt*ddt*mu0 + Gt*ddt*mut + Gt*mu0)/mut,
  N: (-Gt*ddt + Gt)/mut}]

# Usage

1. Ensure you have an xarray dataset ds with the required variables and units (let you be guided by error messages).
2. Apply installation offsets for roll and pitch or determine with find_offsets function in tiltcorrection module.
3. Create a CombinedGlobalDiffuseDirect instance (or just global ... this might not be implemented yet)
4. Call the apply_tilt_correction method. the output is a new CombinedGlobalDiffuseDirect instance with tiltcorrection applied. 

In [ ]:
rin = atmbsr.CombinedGlobalDiffuseDirect(ds, verbose = True)
out = rin.apply_tilt_correction(sensor_response_time=rs)

# sun incidence angel playground

In [1]:
import atmPy.radiation.retrievals.tiltcorrection as atmtilt

/Users/htelg/prog/atm-py/atmPy/radiation/retrievals/tiltcorrection.py:83: SyntaxWarning: invalid escape sequence '\m'
  $N$ the direct normal component, $\mathrm{Diff}$ the diffuse component, $\mu_t$ the


In [4]:
# plain flys in the sun and tilts nose up -> zenith angle gets larger
out = atmtilt.solar_incidence_angle(zenith = np.deg2rad(20), 
                              azimuth = np.deg2rad(90), 
                              pitch = np.deg2rad(1),
                              roll = np.deg2rad(0),
                              heading = np.deg2rad(90))
np.rad2deg(out)

np.float64(20.999999999999986)

In [9]:
# opposite heading
out = atmtilt.solar_incidence_angle(zenith = np.deg2rad(20), 
                              azimuth = np.deg2rad(90), 
                              pitch = np.deg2rad(1),
                              roll = np.deg2rad(0),
                              heading = np.deg2rad(90+180))
np.rad2deg(out)

np.float64(18.999999999999975)

In [8]:
# plain flys in the paralel to the sun and rols right side down (positive roll) -> zenith angle gets smaller
out = atmtilt.solar_incidence_angle(zenith = np.deg2rad(20), 
                              azimuth = np.deg2rad(180), 
                              pitch = np.deg2rad(0),
                              roll = np.deg2rad(1),
                              heading = np.deg2rad(90))
np.rad2deg(out)

np.float64(18.999999999999975)

In [10]:
# oposite heading
out = atmtilt.solar_incidence_angle(zenith = np.deg2rad(20), 
                              azimuth = np.deg2rad(180), 
                              pitch = np.deg2rad(0),
                              roll = np.deg2rad(1),
                              heading = np.deg2rad(90+180))
np.rad2deg(out)

np.float64(20.999999999999986)

np.float64(20.023961889117096)